In [1]:
"""
Name: Jacob Sharon
Date: 11-09-2026
Class: CSC635

This file implements the Feed Forward Neural Network for our groups class project. 

Updates: 
11-09-2026: 
    - Imports, rough outline as TODO cells, set up MLP class for main experiment. 
11-18-2026: 
    - Began preprocessing: reading csv, verifying dtypes, verifying ranges
    - Identified columns from initial read for drop based on clinical utility
11-23-2026:
    - Completed pre-processing: feature selection, missing-value handling, and categorical encoding.
    - Added stratified splitting, standardization, and tensor conversion.
    - Verified input shapes and types; applied positive-class weighting.
    - Updated Markdown documentation.
"""

'\nName: Jacob Sharon\nDate: 11-09-2026\nClass: CSC635\n\nThis file implements the Feed Forward Neural Network for our groups class project. \n\nUpdates: \n11-09-2026: \n    - Imports, rough outline as TODO cells, set up MLP class for main experiment. \n11-18-2026: \n    - Began preprocessing. \n'

In [2]:
# Imports

# Data and Preprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Pytorch 
import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import TensorDataset, DataLoader

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

In [3]:
# Load the dataset

diabetic_data = pd.read_csv("../Data/diabetic_data.csv")

diabetic_data

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


# Preprocessing: Discovery and Feature Selection

I am a Medical Laboratory Scientist and perform high-complexity testing on patients of this demographic. My work is primarily in clinical chemistry, and my previous experience includes supervisory/managerial roles involving billing and ordering using the coding system described in this dataset. I am basing decisions about feature selection on my professional knowledge and understanding of the problem being solved.

Definitions are based on the [UCI dataset documentation](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008).

## Missing Data

Replaced ? and invalid gender entries with NaN. Calculated missing counts and percentages to identify columns with limited recorded data.

In [5]:
# Preprocessing: Identify columns with missing data and quantify missing data.

# Replace all '?' with NaN.
diabetic_data = diabetic_data.replace('?', np.nan)

# Treat invalid gender entries as missing.
diabetic_data['gender'] = diabetic_data['gender'].replace('Unknown/Invalid', np.nan)

# Show missing counts and percentages.
missing_data = pd.DataFrame({
    'missing_count': diabetic_data.isna().sum(),
    'missing_percent': (diabetic_data.isna().mean() * 100).round(2)
})

print(missing_data.to_string())

                          missing_count  missing_percent
encounter_id                          0             0.00
patient_nbr                           0             0.00
race                               2273             2.23
gender                                3             0.00
age                                   0             0.00
weight                            98569            96.86
admission_type_id                     0             0.00
discharge_disposition_id              0             0.00
admission_source_id                   0             0.00
time_in_hospital                      0             0.00
payer_code                        40256            39.56
medical_specialty                 49949            49.08
num_lab_procedures                    0             0.00
num_procedures                        0             0.00
num_medications                       0             0.00
number_outpatient                     0             0.00
number_emergency               

## Medication Use

Medication status indicates whether a drug was prescribed and whether the dosage changed.

- No: drug was not prescribed.
- Steady: dosage did not change.
- Up: dosage increased.
- Down: dosage decreased.

Counted Steady, Up, and Down as medication use. Calculated encounter counts, encounter percentages, and unique patient counts for each drug.

In [7]:
# Preprocessing: Count medication use to help determine which features to retain.

drug_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# Steady, Up, and Down indicate the medication was prescribed.
drug_used = diabetic_data[drug_cols].isin(['Steady', 'Up', 'Down'])

drug_counts = pd.DataFrame({
    'encounters': drug_used.sum(),
    'encounter_percent': (drug_used.mean() * 100).round(2),
    'patients': drug_used.groupby(diabetic_data['patient_nbr']).max().sum()
})

print(drug_counts.sort_values('encounters', ascending=False).to_string())

                          encounters  encounter_percent  patients
insulin                        54383              53.44     39577
metformin                      19988              19.64     16477
glipizide                      12686              12.47      9998
glyburide                      10650              10.47      8558
pioglitazone                    7328               7.20      5939
rosiglitazone                   6365               6.25      5184
glimepiride                     5191               5.10      4187
repaglinide                     1539               1.51      1177
glyburide-metformin              706               0.69       567
nateglinide                      703               0.69       579
acarbose                         308               0.30       236
chlorpropamide                    86               0.08        76
tolazamide                        39               0.04        33
miglitol                          38               0.04        24
tolbutamid

## Feature Selection and Target

- weight: weight in pounds; dropped with 96.86% missing.
- payer_code: identifies the payer; dropped with 39.56% missing.
- medical_specialty: specialty of the admitting physician; dropped with 49.08% missing.
- max_glu_serum: glucose result range or test not measured; dropped with 94.75% unmeasured.
- A1Cresult: HbA1c result range or test not measured; dropped with 83.28% unmeasured.
- encounter_id and patient_nbr: record identifiers; dropped from model inputs.
- diag_2 and diag_3: secondary diagnoses; dropped to reduce dimensionality and focus on the primary diagnosis.
- Medications prescribed in fewer than 100 encounters: dropped due to limited examples, leaving 11 medications.

Converted readmission within 30 days to 1 and later or no recorded readmission to 0. Dropped 21 columns, leaving 28 inputs and one target.

In [9]:
# Preprocessing: Drop columns that are not needed or have too much missing data.

# Keep patient identifiers separately for splitting the data.
patient_ids = diabetic_data['patient_nbr'].copy()

# Drop medications prescribed in fewer than 100 encounters.
drugs_to_drop = [
    'chlorpropamide',
    'tolazamide',
    'miglitol',
    'tolbutamide',
    'glipizide-metformin',
    'troglitazone',
    'metformin-rosiglitazone',
    'acetohexamide',
    'glimepiride-pioglitazone',
    'metformin-pioglitazone',
    'examide',
    'citoglipton'
]

diabetic_data.drop(
    columns=[
        'encounter_id', 
        'patient_nbr', 
        'weight', 
        'payer_code',
        'medical_specialty', 
        'max_glu_serum', 
        'A1Cresult', 
        'diag_2', 
        'diag_3'
    ] + drugs_to_drop,
    inplace=True
)

# Preprocessing: Convert readmission within 30 days to a binary target.

diabetic_data['readmitted'] = (
    diabetic_data['readmitted'] == '<30'
).astype(int)

print(diabetic_data['readmitted'].value_counts())

readmitted
0    90409
1    11357
Name: count, dtype: int64


## Primary Diagnosis

diag_1: primary diagnosis recorded as an ICD-9 code.

Kept entries containing exactly three digits and assigned missing entries and codes outside this rule to -1. Converted the column to float32 and printed the counts for each value. The rule assigned 12,834 entries to -1; valid codes were retained regardless of how often they occurred. 

TODO: There are several diagnosis codes only used once, these may be combined later depending on model performance. 

In [11]:
# Preprocessing: Retain three-digit diagnosis codes.
# Assign all other entries -1 to represent missing or excluded codes.

diagnosis = diabetic_data['diag_1'].astype('string')

valid_diagnosis = diagnosis.str.fullmatch(r'[0-9]{3}', na=False)

diabetic_data['diag_1'] = pd.to_numeric(
    diagnosis.where(valid_diagnosis),
    errors='coerce'
).fillna(-1).astype('float32')

print(diabetic_data['diag_1'].value_counts())

diag_1
-1.0      12834
 428.0     6862
 414.0     6581
 786.0     4016
 410.0     3614
          ...  
 314.0        1
 684.0        1
 817.0        1
 640.0        1
 690.0        1
Name: count, Length: 628, dtype: int64


## Categorical Missing Values

Converted admission type, discharge disposition, and admission source IDs to strings so they are handled as categories. Identified all text-based categorical columns and replaced their missing entries with a separate Missing category. No encounters were dropped in this step.

In [13]:
# Preprocessing: Treat diagnosis codes and category IDs as categorical features.

coded_cols = [
    'admission_type_id', 
    'discharge_disposition_id', 
    'admission_source_id'
]

diabetic_data[coded_cols] = diabetic_data[coded_cols].astype('string')

categorical_cols = diabetic_data.select_dtypes(
    include=['object', 'string']
).columns.tolist()

# Replace missing categorical values with a separate category.
diabetic_data[categorical_cols] = diabetic_data[categorical_cols].fillna('Missing')

## Categorical Encoding

Identified the numerical input columns before encoding and excluded the target from that list. Used OrdinalEncoder to assign numerical labels to categorical values, keeping one column per feature. The encoded labels are stored as float32 and introduce numerical ordering between categories.

In [15]:
# Preprocessing: Encode categorical features without adding columns.

from sklearn.preprocessing import OrdinalEncoder

# Identify numerical features before encoding categories.
numeric_cols = diabetic_data.select_dtypes(
    include=['number']
).columns.drop('readmitted').tolist()

encoder = OrdinalEncoder(dtype=np.float32)
diabetic_data[categorical_cols] = encoder.fit_transform(
    diabetic_data[categorical_cols]
)

print("Dataset shape after encoding:", diabetic_data.shape)

Dataset shape after encoding: (101766, 29)


## Train/Test Split

Separated the inputs from the readmission target and used an 80/20 stratified random split by encounter. Printed class counts and percentages for 81,412 training rows and 20,354 testing rows. Both sets retain approximately 11.16% positive and 88.84% negative cases; stratification preserves these proportions.

In [17]:
# Preprocessing: Separate features and target.

X = diabetic_data.drop(columns=['readmitted'])
y = diabetic_data['readmitted']

# Preserve the target class proportions when splitting.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check the class counts and percentages.
class_counts = pd.DataFrame({
    'training_count': y_train.value_counts(),
    'testing_count': y_test.value_counts(),
    'training_percent': y_train.value_counts(normalize=True).mul(100),
    'testing_percent': y_test.value_counts(normalize=True).mul(100)
})

print(class_counts.sort_index().round(2).to_string())

            training_count  testing_count  training_percent  testing_percent
readmitted                                                                  
0                    72326          18083             88.84            88.84
1                     9086           2271             11.16            11.16


## Standardization and Input Checks

Fitted StandardScaler on the original numerical inputs in the training set, including diag_1, and applied the same transformation to the test set. The -1 diagnosis value is standardized with the rest of that column. Encoded categorical labels remain unscaled. Converted all inputs to float32 and printed the final shapes and missing counts: 28 inputs in both sets, with no missing values.

In [19]:
# Preprocessing: Standardize numerical features using training data.

X_train = X_train.copy()
X_test = X_test.copy()

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Convert all input features to float32 for PyTorch.
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Missing training values:", X_train.isna().sum().sum())
print("Missing testing values:", X_test.isna().sum().sum())

Training shape: (81412, 28)
Testing shape: (20354, 28)
Missing training values: 0
Missing testing values: 0


## Input Shapes and Data Types

Printed the shapes and data types of X and y, followed by the training and testing inputs. X is the encoded data before standardization; X_train and X_test are the prepared inputs used for training and evaluation.

In [21]:
# Preprocessing: Check input and target shapes and data types.

print("X shape:", X.shape)
print("X column dtypes:")
print(X.dtypes.to_string())
print("y shape:", y.shape, "dtype:", y.dtype)

print("X_train:", X_train.shape, "dtypes:", X_train.dtypes.unique())
print("X_test:", X_test.shape, "dtypes:", X_test.dtypes.unique())
print("y_train:", y_train.shape, "dtype:", y_train.dtype)
print("y_test:", y_test.shape, "dtype:", y_test.dtype)

X shape: (101766, 28)
X column dtypes:
race                        float32
gender                      float32
age                         float32
admission_type_id           float32
discharge_disposition_id    float32
admission_source_id         float32
time_in_hospital              int64
num_lab_procedures            int64
num_procedures                int64
num_medications               int64
number_outpatient             int64
number_emergency              int64
number_inpatient              int64
diag_1                      float32
number_diagnoses              int64
metformin                   float32
repaglinide                 float32
nateglinide                 float32
glimepiride                 float32
glipizide                   float32
glyburide                   float32
pioglitazone                float32
rosiglitazone               float32
acarbose                    float32
insulin                     float32
glyburide-metformin         float32
change                   

## Tensor Conversion

Converted the prepared training and testing inputs to float32 tensors. Converted the binary targets to float32 tensors with one column so their shapes match the MLP output for BCEWithLogitsLoss. Printed each tensor's shape and data type.

In [23]:
# Preprocessing: Convert inputs and targets to PyTorch tensors.

X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)

# BCEWithLogitsLoss requires float targets matching the output shape.
y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

print("X_train_tensor:", X_train_tensor.shape, X_train_tensor.dtype)
print("X_test_tensor:", X_test_tensor.shape, X_test_tensor.dtype)
print("y_train_tensor:", y_train_tensor.shape, y_train_tensor.dtype)
print("y_test_tensor:", y_test_tensor.shape, y_test_tensor.dtype)

X_train_tensor: torch.Size([81412, 28]) torch.float32
X_test_tensor: torch.Size([20354, 28]) torch.float32
y_train_tensor: torch.Size([81412, 1]) torch.float32
y_test_tensor: torch.Size([20354, 1]) torch.float32


In [24]:
# Neural Network Implementation

class MLP(nn.Module):
    """
    A fully connected feedforward neural network (Multi-Layer Perceptron)
    used for binary classification of early readmission of diabteic patients
    within 30 days of discharge.

    This implementation supports an arbitrary number of hidden layers, each
    followed by:
      • Batch Normalization (except after the first layer)
      • Dropout (p = 0.02)
      • ReLU activation

    Kaiming uniform initialization is applied to all Linear layer weights,
    and biases are initialized to zero. The final layer outputs an unbounded
    logit suitable for use with BCEWithLogitsLoss.

    Parameters
    ----------
    input_size : int
        Number of input features.

    hidden_sizes : list of int
        A list specifying the width of each hidden layer.
        Example: [64, 32] produces a network with two hidden layers.

    output_size : int
        Number of output neurons. For binary classification, this is 1.

    Attributes
    ----------
    network : nn.Sequential
        The sequential model containing all layers in forward order.
    """

    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()

        layers = []

        # First hidden layer (no batch norm / dropout here)
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())

        # Remaining hidden layers
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            layers.append(nn.BatchNorm1d(hidden_sizes[i + 1]))
            layers.append(nn.Dropout(p=0.02))
            layers.append(nn.ReLU())

        # Output layer (produces logits)
        layers.append(nn.Linear(hidden_sizes[-1], output_size))

        # Construct sequential model
        self.network = nn.Sequential(*layers)

        # Initialize parameters
        self._init_weights()

    def _init_weights(self):
        """
        Initialize weights of all Linear layers using Kaiming uniform
        initialization, which is recommended for ReLU-activated networks.

        Bias terms are initialized to zero.
        """
        for layer in self.network:
            if isinstance(layer, nn.Linear):
                init.kaiming_uniform_(layer.weight, nonlinearity='relu')
                if layer.bias is not None:
                    init.zeros_(layer.bias)

    def forward(self, x):
        """
        Perform a forward pass through the network.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (batch_size, input_size).

        Returns
        -------
        torch.Tensor
            Output logits of shape (batch_size, output_size).
            These logits should be passed to sigmoid() or directly to
            BCEWithLogitsLoss for stable binary classification.
        """
        return self.network(x)

## Loss Function and Class Imbalance

Used BCEWithLogitsLoss with positive-class weighting to account for the smaller number of early readmission cases. Calculated the weight using training targets only: 72,326 negative cases divided by 9,086 positive cases, approximately 7.96.

Applied this ratio through pos_weight to increase the positive-class contribution to the loss while retaining all training encounters. The MLP supplies raw logits, and the targets are float32 column tensors containing 0 or 1. Printed the counts, calculated ratio, and loss weight tensor for confirmation.

Reference: [PyTorch BCEWithLogitsLoss documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html).

In [26]:
# Loss function: Account for class imbalance using training targets only.

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

# Weight the positive-class loss by the negative-to-positive training ratio.
ratio = negative_count / positive_count

# Keep the weight on the same device and dtype as the target tensors.
pos_weight = torch.tensor(
    [ratio], dtype=y_train_tensor.dtype, device=y_train_tensor.device
)

# Pass raw MLP logits directly to BCEWithLogitsLoss.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Negative training cases:", negative_count)
print("Positive training cases:", positive_count)
print(f"Positive class weight: {ratio:.4f}")
print("Loss weight tensor:", criterion.pos_weight)
print(criterion)

Negative training cases: 72326
Positive training cases: 9086
Positive class weight: 7.9602
Loss weight tensor: tensor([7.9602])
BCEWithLogitsLoss()


In [27]:
# Experimental Designs and Hyperparameter Tuning

"""
TODO: 
    - Begin with using all input features and then perform feature selection to narrow. 
    - Define network architectures to test, probably keep limited to two hidden layers for this class.
    - Decide which metrics to collect and best representation. 
"""

'\nTODO: \n    - Begin with using all input features and then perform feature selection to narrow. \n    - Define network architectures to test, probably keep limited to two hidden layers for this class.\n    - Decide which metrics to collect and best representation. \n'

In [28]:
# Data Collection and Plotting